In [45]:
PROCESSED_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/FH/processed-files"
DATA_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/FH/data"

In [46]:
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict
import pickle
from lenskit import topn

In [47]:
train=pd.read_csv(f"{DATA_FILE_DIR}/train_ratings.csv").iloc[:,[1,2,3]]

In [48]:
train

,user,item,ratings
0,0,624,1.0
1,0,484,1.5
2,0,466,1.5
3,0,377,1.5
4,0,371,1.5
...,...,...,...
114715,5405,19,1.5
114716,5405,4,1.5
114717,5405,114,1.5
114718,5406,2,1.5


In [49]:
train.drop_duplicates(subset=['user', 'item'], inplace=True)


In [50]:
user_item_sets = train.groupby('user')['item'].apply(set).to_dict()

In [51]:
user_item_sets

{0: {160, 193, 241, 248, 261, 266, 336, 371, 377, 466, 484, 624},
 1: {77, 79, 106, 138, 174, 195, 229, 302, 319, 338, 378},
 2: {787, 791, 795, 798, 801, 805, 807, 813, 816, 822, 824},
 3: {261,
  351,
  391,
  503,
  527,
  533,
  600,
  624,
  646,
  657,
  659,
  669,
  695,
  724,
  739,
  1012,
  1024,
  1028,
  1092,
  1104,
  1142,
  1324,
  1334,
  1400,
  1450,
  1515,
  1562,
  1604,
  1635,
  1667},
 4: {163, 165, 204, 262, 264, 374, 443, 692},
 5: {440,
  600,
  604,
  616,
  624,
  646,
  657,
  695,
  859,
  983,
  1028,
  1102,
  1128,
  1154,
  1293,
  1300,
  1310,
  1334,
  1360,
  1370,
  1377,
  1382,
  1400,
  1419,
  1444,
  1449,
  1450,
  1494,
  1515,
  1519,
  1562,
  1659,
  1679},
 6: {64, 90, 381, 434, 889, 1061, 1283, 1327, 1363, 1432, 1524},
 7: {657, 1012, 1024, 1114, 1300, 1304, 1543, 1552, 1604},
 8: {351,
  495,
  527,
  584,
  600,
  604,
  616,
  631,
  646,
  1028,
  1092,
  1104,
  1206,
  1284,
  1304,
  1318,
  1334,
  1377,
  1400,
  1515,
  1

In [52]:
def calculate_cosine_similarity(user1, user2, user_item_sets):
    items_user1 = user_item_sets.get(user1, set())
    items_user2 = user_item_sets.get(user2, set())
    
    # Calculate intersection of items
    intersection = len(items_user1.intersection(items_user2))
    
    # Compute the dot product of the vectors representing the item sets
    dot_product = intersection
    
    # Compute the magnitudes of the vectors
    magnitude_user1 = len(items_user1)
    magnitude_user2 = len(items_user2)
    
    # Handle division by zero
    if magnitude_user1 == 0 or magnitude_user2 == 0:
        return 0.0
    
    # Compute cosine similarity
    cosine_similarity = dot_product / (magnitude_user1 * magnitude_user2) ** 0.5
    return cosine_similarity



In [53]:
unique_users = train['user'].unique()

In [54]:
# Calculate Jaccard similarity for all user pairs
cosine_similarities = []

for user1, user2 in product(unique_users, repeat=2):
    if user1 != user2:
        similarity = calculate_cosine_similarity(user1, user2,user_item_sets)
        cosine_similarities.append({'User1': user1, 'User2': user2, 'Similarity': similarity})

In [55]:
# Create a DataFrame for Jaccard similarities
cosine_df = pd.DataFrame(cosine_similarities)

# Reshape the DataFrame to have User1, User2, and Similarity as columns
cosine_matrix = cosine_df.pivot(index='User1', columns='User2', values='Similarity')

idx = range(len(cosine_matrix))
cosine_matrix.values[idx, idx] = 1.0


In [56]:
cosine_matrix

User2,0,1,2,3,4,5,6,7,8,9,...,5397,5398,5399,5400,5401,5402,5403,5404,5405,5406
User1,,,,,,,,,,,,,,,,,,,,,
0,1.000000,0.0,0.0,0.105409,0.0,0.050252,0.000000,0.000000,0.000000,0.125245,...,0.068041,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
1,0.000000,1.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.083624,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
2,0.000000,0.0,1.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
3,0.105409,0.0,0.0,1.000000,0.0,0.349603,0.000000,0.243432,0.428174,0.534680,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
4,0.000000,0.0,0.0,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.083333,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5402,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.083333,0.294174,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.0,0.0
5403,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.087039,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.577350,0.696311,0.654654,0.0,1.000000,0.436436,0.0,0.0
5404,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.503953,0.569803,0.571429,0.0,0.436436,1.000000,0.0,0.0


In [57]:
def file_load(path):
    my_file = open(path, "r")
    # reading the file
    data = my_file.read()
    # splitting the text it further when '.' is seen.
    data_into_list = data.split("\n")[:-1]
    myseed=[int(item) for item in data_into_list]
    return myseed


In [58]:
superspreader=file_load(f'{DATA_FILE_DIR}/superspreader_50.txt')

In [59]:
# Create untrusted users list
untrusted_users = list(superspreader)

# Remove user 0 if it exists (consistent with your Politifact code)
if 0 in untrusted_users:
    untrusted_users.remove(0)

# Save to pickle
with open(f'{PROCESSED_FILE_DIR}/untrusted_users_ss50.pickle', 'wb') as f:
    pickle.dump(untrusted_users, f)

print(f"Saved {len(untrusted_users)} untrusted users to pickle file")

Saved 281 untrusted users to pickle file


In [60]:
reputation_threshold = 0.6

# Filter users based on reputation greater than 0.6 and not in superspreaders
trusted_users = [i for i in range(5407) if i not in superspreader]

# Dictionary to store top similar trustworthy users for each user
top_trustworthy_similar_users = defaultdict(set)

# Iterate over each user
for user in cosine_matrix.index:
    # Get cosine similarity values for the user
    user_similarity = cosine_matrix.loc[user]
    # Exclude untrustworthy similar users (not in trusted_users), users in superspreaders, and the user itself
    similar_trustworthy_users = user_similarity[(user_similarity > 0) & (user_similarity.index != user) & (~np.isin(user_similarity.index, superspreader)) & (user_similarity.index.isin(trusted_users))]
    # Sort similar trustworthy users by similarity (descending order) and take the top 10
    temp = set(similar_trustworthy_users.sort_values(ascending=False).index[:10].tolist())
    if len(temp) > 0:
        top_trustworthy_similar_users[user] = temp


In [61]:
top_trustworthy_similar_users

defaultdict(set,
            {0: {977, 1300, 1343, 1623, 1681, 4227, 4858, 4888, 4976, 5205},
             1: {2902, 3160, 3647, 3824, 4589, 4642, 4832, 5165, 5225, 5345},
             2: {769, 3295, 4133, 4153, 4260, 4309, 4334, 4394, 4452, 4458},
             3: {9, 420, 744, 1431, 1489, 1572, 2626, 3395, 4794, 4995},
             4: {253, 610, 716, 1156, 1690, 4641, 4787, 5048, 5113, 5114},
             5: {9, 789, 792, 840, 844, 1122, 1333, 2056, 2812, 3068},
             6: {54, 245, 587, 1594, 1644, 1709, 2159, 2699, 2706, 2786},
             7: {888, 1182, 1293, 1375, 1572, 1697, 1927, 2452, 3375, 3883},
             8: {9, 470, 744, 1107, 1526, 2060, 2182, 2291, 2580, 2626},
             9: {789, 792, 840, 844, 888, 1122, 1132, 1145, 1333, 1431},
             10: {518, 610, 876, 3194, 4542, 4655, 4972, 5183, 5288, 5350},
             11: {551, 1164, 1478, 1532, 3631, 4422, 4547, 4702, 4703, 4793},
             12: {1042, 1206, 2239, 2335, 2542, 2603, 2716, 4101, 4190, 4507},
  

In [62]:
# Specify the path to your pickle file
pickle_file_path = f"{PROCESSED_FILE_DIR}/TrustWorthy_FH_MAGrec.pickle"

# Open the pickle file in read binary mode
with open(pickle_file_path, 'rb') as file:
    # Load the data from the pickle file
    loaded_data = pickle.load(file)

In [63]:
history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list=loaded_data

In [64]:
social_adj_lists=top_trustworthy_similar_users

In [65]:
social_adj_lists

defaultdict(set,
            {0: {977, 1300, 1343, 1623, 1681, 4227, 4858, 4888, 4976, 5205},
             1: {2902, 3160, 3647, 3824, 4589, 4642, 4832, 5165, 5225, 5345},
             2: {769, 3295, 4133, 4153, 4260, 4309, 4334, 4394, 4452, 4458},
             3: {9, 420, 744, 1431, 1489, 1572, 2626, 3395, 4794, 4995},
             4: {253, 610, 716, 1156, 1690, 4641, 4787, 5048, 5113, 5114},
             5: {9, 789, 792, 840, 844, 1122, 1333, 2056, 2812, 3068},
             6: {54, 245, 587, 1594, 1644, 1709, 2159, 2699, 2706, 2786},
             7: {888, 1182, 1293, 1375, 1572, 1697, 1927, 2452, 3375, 3883},
             8: {9, 470, 744, 1107, 1526, 2060, 2182, 2291, 2580, 2626},
             9: {789, 792, 840, 844, 888, 1122, 1132, 1145, 1333, 1431},
             10: {518, 610, 876, 3194, 4542, 4655, 4972, 5183, 5288, 5350},
             11: {551, 1164, 1478, 1532, 3631, 4422, 4547, 4702, 4703, 4793},
             12: {1042, 1206, 2239, 2335, 2542, 2603, 2716, 4101, 4190, 4507},
  

In [66]:
import pickle
# Organize the variables into a tuple in the desired order
data_to_save = (history_u_lists, history_ur_lists, history_v_lists, history_vr_lists,
                train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list)

# Specify the path to save the pickle file
output_pickle_file_path = f"{PROCESSED_FILE_DIR}/FH_Trustworthy_Neighbors_0.6_threshold.pickle"

# Open the pickle file in write binary mode ('wb')
with open(output_pickle_file_path, 'wb') as file:
    # Dump the data to the pickle file
    pickle.dump(data_to_save, file)

print("Data saved to pickle file:", output_pickle_file_path)


Data saved to pickle file: /home/shoaib/recommender-system/royal/journal-revision/New/FH/processed-files/FH_Trustworthy_Neighbors_0.6_threshold.pickle


## Training

In [67]:
import torch
import torch.nn as nn
import pickle
import numpy as np
import argparse
import os
from UV_Encoders import UV_Encoder
from UV_Aggregators import UV_Aggregator
from Social_Encoders import Social_Encoder
from Social_Aggregators import Social_Aggregator


class GraphRec(nn.Module):
    def __init__(self, enc_u, enc_v_history, r2e):
        super(GraphRec, self).__init__()
        self.enc_u = enc_u
        self.enc_v_history = enc_v_history
        self.embed_dim = enc_u.embed_dim
        self.w_ur1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_ur2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_uv1 = nn.Linear(self.embed_dim * 2, self.embed_dim)
        self.w_uv2 = nn.Linear(self.embed_dim, 16)
        self.w_uv3 = nn.Linear(16, 1)
        self.r2e = r2e
        self.bn1 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn2 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn3 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn4 = nn.BatchNorm1d(16, momentum=0.5)

    def forward(self, nodes_u, nodes_v):
        embeds_u = self.enc_u(nodes_u)
        embeds_v = self.enc_v_history(nodes_v)
        scores_u = torch.matmul(embeds_u, embeds_v.t())
        return scores_u.squeeze()

    def bpr_loss(self, nodes_u, nodes_v, device, num_items):
        scores_u = self.forward(nodes_u, nodes_v)
        nodes_v_negative = torch.randint(0, num_items, size=nodes_u.size(), dtype=torch.long).to(device)
        scores_u_negative = self.forward(nodes_u, nodes_v_negative)
        bpr_loss = -torch.log(torch.sigmoid(scores_u - scores_u_negative)).mean()
        return bpr_loss

def train(model, device, train_loader, optimizer, epoch, num_items):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        batch_nodes_u, batch_nodes_v, _ = data
        optimizer.zero_grad()
        loss = model.bpr_loss(batch_nodes_u.to(device), batch_nodes_v.to(device), device, num_items)
        loss.backward(retain_graph=True)
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 0:
            print('[%d, %5d] loss: %.3f' % (epoch, i, running_loss / 100))
            running_loss = 0.0

def get_top_100_recommendations(model, device, test_loader, num_items, k=100):
    model.eval()
    user_recommendations_dict = {}  # Dictionary to store user recommendations

    with torch.no_grad():
        for test_u, test_v, _ in test_loader:
            test_u, test_v = test_u.to(device), test_v.to(device)
            scores = model.forward(test_u, test_v)
            scores = scores.squeeze().tolist()
            top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

            user_ids = test_u.tolist()  # Convert tensor to list

            for user_id in user_ids:
                user_recommendations = [(rank, test_v[item_index].item()) for rank, item_index in enumerate(top_indices, start=1)]
                user_recommendations_dict[user_id] = user_recommendations

    return user_recommendations_dict




def main():
    parser = argparse.ArgumentParser(description='Social Recommendation: GraphRec model')
    parser.add_argument('--batch_size', type=int, default=128, metavar='N', help='input batch size for training')
    parser.add_argument('--embed_dim', type=int, default=64, metavar='N', help='embedding size')
    parser.add_argument('--lr', type=float, default=0.001, metavar='LR', help='learning rate')
    parser.add_argument('--test_batch_size', type=int, default=1000, metavar='N', help='input batch size for testing')
    parser.add_argument('--epochs', type=int, default=100, metavar='N', help='number of epochs to train')
    args, unknown = parser.parse_known_args()

    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    use_cuda = False
    if torch.cuda.is_available():
        use_cuda = True
    device = torch.device("cuda" if use_cuda else "cpu")

    embed_dim = args.embed_dim
    dir_data = f'{PROCESSED_FILE_DIR}/FH_Trustworthy_Neighbors_0.6_threshold'

    path_data = dir_data + ".pickle"
    data_file = open(path_data, 'rb')
    history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list = pickle.load(
        data_file)
    total_num_items = len(set(train_v + test_v))

    trainset = torch.utils.data.TensorDataset(torch.LongTensor(train_u), torch.LongTensor(train_v),
                                              torch.FloatTensor(train_r))
    testset = torch.utils.data.TensorDataset(torch.LongTensor(test_u), torch.LongTensor(test_v),
                                             torch.FloatTensor(test_r))
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=True)
    num_users = history_u_lists.__len__()
    num_items = history_v_lists.__len__()

    u2e = nn.Embedding(num_users, embed_dim).to(device)
    v2e = nn.Embedding(num_items, embed_dim).to(device)
    r2e = nn.Embedding(total_num_items, embed_dim).to(device)  # Define r2e with the correct number of ratings
    total_num_items = len(set(train_v + test_v))

    agg_u_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=True)
    enc_u_history = UV_Encoder(u2e, embed_dim, history_u_lists, history_ur_lists, agg_u_history, cuda=device, uv=True)
    agg_u_social = Social_Aggregator(lambda nodes: enc_u_history(nodes).t(), u2e, embed_dim, cuda=device)
    enc_u = Social_Encoder(lambda nodes: enc_u_history(nodes).t(), embed_dim, social_adj_lists, agg_u_social,
                           base_model=enc_u_history, cuda=device)

    agg_v_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=False)
    enc_v_history = UV_Encoder(v2e, embed_dim, history_v_lists, history_vr_lists, agg_v_history, cuda=device, uv=False)

    graphrec = GraphRec(enc_u, enc_v_history, r2e).to(device)
    optimizer = torch.optim.RMSprop(graphrec.parameters(), lr=args.lr, alpha=0.9)

    for epoch in range(1, args.epochs + 1):
        train(graphrec, device, train_loader, optimizer, epoch, total_num_items)

    top_100_recommendations = get_top_100_recommendations(graphrec, device, test_loader, num_items=total_num_items)

    # Print or use top_100_recommendations as needed
    return top_100_recommendations

if __name__ == "__main__":
    recommendation_list=main()


[1,     0] loss: 0.007
[1,   100] loss: 0.543
[1,   200] loss: 0.515
[1,   300] loss: 0.517
[1,   400] loss: 0.509
[1,   500] loss: 0.508
[1,   600] loss: 0.509
[1,   700] loss: 0.511
[1,   800] loss: 0.519
[2,     0] loss: 0.005
[2,   100] loss: 0.510
[2,   200] loss: 0.512
[2,   300] loss: 0.508
[2,   400] loss: 0.506
[2,   500] loss: 0.510
[2,   600] loss: 0.506
[2,   700] loss: 0.499
[2,   800] loss: 0.497
[3,     0] loss: 0.005
[3,   100] loss: 0.509
[3,   200] loss: 0.503
[3,   300] loss: 0.516
[3,   400] loss: 0.505
[3,   500] loss: 0.503
[3,   600] loss: 0.505
[3,   700] loss: 0.499
[3,   800] loss: 0.507
[4,     0] loss: 0.005
[4,   100] loss: 0.501
[4,   200] loss: 0.504
[4,   300] loss: 0.493
[4,   400] loss: 0.501
[4,   500] loss: 0.499
[4,   600] loss: 0.495
[4,   700] loss: 0.499
[4,   800] loss: 0.506
[5,     0] loss: 0.005
[5,   100] loss: 0.500
[5,   200] loss: 0.497
[5,   300] loss: 0.502
[5,   400] loss: 0.497
[5,   500] loss: 0.492
[5,   600] loss: 0.505
[5,   700] 

In [68]:
file_path = f'{PROCESSED_FILE_DIR}/FH_item_update.pickle'

with open(file_path, 'rb') as file:
    loaded_dictionary = pickle.load(file)

In [69]:
def get_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

In [70]:
rows = []
for user, rankings in recommendation_list.items():
    for rank, item in rankings:
        rows.append({"user": user, "rank": rank, "item_update": item, "item": get_key_by_value(loaded_dictionary, item)})

all_recs = pd.DataFrame(rows)

In [71]:
cold_start_item=[2,571]

In [72]:
filtered_recs = all_recs[~all_recs['item'].isin(cold_start_item)]
filtered_recs

,user,rank,item_update,item
0,3411,1,318,324
1,3411,2,256,262
2,3411,3,923,931
3,3411,4,1179,1188
4,3411,5,1308,1319
...,...,...,...,...
540695,3605,96,339,345
540696,3605,97,141,142
540697,3605,98,641,649
540698,3605,99,249,255


In [73]:
filtered_recs['rank'] = filtered_recs['rank'].astype(int)

/tmp/ipykernel_1399667/435666994.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['rank'] = filtered_recs['rank'].astype(int)


In [74]:
filtered_recs['re-rank'] = filtered_recs.groupby('user')['rank'].rank(method='first')
filtered_recs['rank'] = filtered_recs['re-rank'].astype(int)
filtered_recs=filtered_recs.iloc[:,[0,1,2,3]]

/tmp/ipykernel_1399667/347894999.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['re-rank'] = filtered_recs.groupby('user')['rank'].rank(method='first')
/tmp/ipykernel_1399667/347894999.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['rank'] = filtered_recs['re-rank'].astype(int)


In [75]:
# Save filtered_recs to pickle
filtered_recs.to_pickle(f'{PROCESSED_FILE_DIR}/FH_filtered_recs_neighbour_0.6_threshold.pickle')

In [76]:
filtered_recs

,user,rank,item_update,item
0,3411,1,318,324
1,3411,2,256,262
2,3411,3,923,931
3,3411,4,1179,1188
4,3411,5,1308,1319
...,...,...,...,...
540695,3605,96,339,345
540696,3605,97,141,142
540697,3605,98,641,649
540698,3605,99,249,255


In [77]:
# Define the different top-k values you want to generate
top_k_values = [5, 10, 15, 20]

# Loop through each k value
for k in top_k_values:
    # Filter for top-k recommendations (excluding user 0)
    MA_gnn_all_recs_top_k = filtered_recs[(filtered_recs['rank'] <= k) & (filtered_recs['user']!=5406)]
    
    # Select only the required columns: user, rank, item
    final_all_recs = MA_gnn_all_recs_top_k.iloc[:, [0, 1, 3]]
    
    # Display the final DataFrame (with only 3 columns)
    print(f"\n{'='*60}")
    print(f"TOP-{k} RECOMMENDATIONS (user, rank, item)")
    print(f"{'='*60}")
    display(final_all_recs)
    
    # Save to CSV with k in the filename
    filename = f'{DATA_FILE_DIR}/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_{k}.csv'
    final_all_recs.to_csv(filename, index=False)
    
    print(f"\n✓ Saved {len(final_all_recs)} recommendations to {filename}")
    print(f"  - Number of users: {final_all_recs['user'].nunique()}")


TOP-5 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,3411,1,324
1,3411,2,262
2,3411,3,931
3,3411,4,1188
4,3411,5,1319
...,...,...,...
540600,3605,1,290
540601,3605,2,762
540602,3605,3,1022
540603,3605,4,1076



✓ Saved 27030 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_5.csv
  - Number of users: 5406

TOP-10 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,3411,1,324
1,3411,2,262
2,3411,3,931
3,3411,4,1188
4,3411,5,1319
...,...,...,...
540605,3605,6,733
540606,3605,7,1
540607,3605,8,8
540608,3605,9,293



✓ Saved 54060 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_10.csv
  - Number of users: 5406

TOP-15 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,3411,1,324
1,3411,2,262
2,3411,3,931
3,3411,4,1188
4,3411,5,1319
...,...,...,...
540610,3605,11,840
540611,3605,12,1178
540612,3605,13,1206
540613,3605,14,339



✓ Saved 81090 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_15.csv
  - Number of users: 5406

TOP-20 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,3411,1,324
1,3411,2,262
2,3411,3,931
3,3411,4,1188
4,3411,5,1319
...,...,...,...
540615,3605,16,908
540616,3605,17,193
540617,3605,18,140
540618,3605,19,328



✓ Saved 108120 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_20.csv
  - Number of users: 5406


## Evaluation

In [78]:
test=pd.read_csv(f"{DATA_FILE_DIR}/test_ratings_final_before_train.csv")
test=test.iloc[:,[1,2,3]]
test.drop(index=5406,inplace=True)
test

,user,item,ratings
0,0,332,1.0
1,1,66,1.0
2,2,785,1.5
3,3,85,1.5
4,4,120,1.5
...,...,...,...
5401,5401,1,0.5
5402,5402,5,2.0
5403,5403,1,1.0
5404,5404,1,1.5


In [79]:
top_k_values = [5, 10, 15, 20]

# Initialize RecListAnalysis
rla = topn.RecListAnalysis()
rla.add_metric(topn.recip_rank)

# Store results
mrr_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_{k}.csv")
    
    # Compute MRR
    results = rla.compute(final_all_recs, test)
    mean_results = results.mean()
    
    # Check what's in the results
    print(f"\nTop-{k} Results:")
    print(mean_results)
    
    mrr_results[k] = mean_results['recip_rank']

# Create summary DataFrame
mrr_summary = pd.DataFrame(list(mrr_results.items()), columns=['Top-K', 'MRR'])
display(mrr_summary)


Top-5 Results:
nrecs         5.00000
recip_rank    0.00792
dtype: float64

Top-10 Results:
nrecs         10.000000
recip_rank     0.010056
dtype: float64

Top-15 Results:
nrecs         15.000000
recip_rank     0.011126
dtype: float64

Top-20 Results:
nrecs         20.000000
recip_rank     0.012528
dtype: float64


,Top-K,MRR
0,5,0.007920
1,10,0.010056
2,15,0.011126
3,20,0.012528


In [80]:
# Load the key (only once)
key = pd.read_csv(f"{DATA_FILE_DIR}/key_healthstory.csv")
key_news_id = key[["item","label"]].drop_duplicates(subset=["item"], keep="last").reset_index(drop=True)
key_news_id["fake_real_labels"] = key_news_id["label"].apply(lambda x: 0 if x == "fake" else 1)

# Define the MMC function
def mmc(dataframe, top_n):
    """
    Calculates the Misinformation Count (MC) divided by the top-N recommendations for each user.

    Parameters:
    dataframe (DataFrame): DataFrame with columns 'user_id', 'item_id', 'rank', 'news_label'.
    top_n (int): Number of top recommendations to consider.

    Returns:
    float: Mean MC across all users.
    """
    dataframe = dataframe[dataframe['rank'] <= top_n]
    sorted_df = dataframe.sort_values(by=['user', 'rank'])
    misinformation_df = sorted_df[sorted_df['label'] == 'fake']
    user_mc = misinformation_df.groupby('user').size().reset_index(name='M_item')
    user_mc['MC'] = user_mc['M_item'] / top_n
    return user_mc['MC'].mean()

# Compute MC for all top-k values
top_k_values = [5, 10, 15, 20]
mc_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/FH_Trustworthy_Neighbor_Space_0.6_threshold_top_{k}.csv")
    
    # Merge with news labels
    final_recs_mc = pd.merge(final_all_recs, key_news_id, left_on="item", right_on="item", how='inner')
    
    # Compute MC for this top-k
    mc_score = mmc(final_recs_mc, k)
    mc_results[k] = mc_score
    
    print(f"Top-{k} MC: {mc_score:.4f}")

# Create summary DataFrame
mc_summary = pd.DataFrame(list(mc_results.items()), columns=['Top-K', 'MC'])
display(mc_summary)

Top-5 MC: 0.4400
Top-10 MC: 0.3664
Top-15 MC: 0.3429
Top-20 MC: 0.3312


,Top-K,MC
0,5,0.439968
1,10,0.366445
2,15,0.342940
3,20,0.331188
